# GRD Severity & Insurance Analysis — Line B (C1)

**Group 03 — Miguel Figueroa, Sebastián Bunzli**

Research question: among privately-insured patients (Isapre/Particular) who
end up treated in Chile's **public** hospital network, is their clinical
severity profile different from FONASA beneficiaries? See the refinement
note further below for why the question is framed this way and what it can
and cannot show.

**Before running:** update `DATA_ROOT` in the next code cell to match where
the course's GRD files are stored on your machine.

**Notebook structure:**
1. Setup
2. Data quality exploration (Five Cs checks on the raw files)
3. Cleaning pipeline
4. Research question note
5. KPI calculation across years (2021-2024)
6. Interpretation of the 2021 anomaly
7. Visualizations
8. Export reduced input/output tables


In [1]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # remove this line for inline plots in Jupyter
import matplotlib.pyplot as plt
from pathlib import Path


## 1. Configuration

In [2]:
# Update this single path if running on a different machine.
DATA_ROOT = r"C:\Users\sebal\Downloads\BI_T-main"

# 2022 is published by FONASA as "GRD_PUBLICO_EXTERNO" instead of
# "GRD_PUBLICO", and 2022/2023 are UTF-16 encoded while 2021/2024 are
# latin1. Both facts were confirmed by direct byte/header inspection
# (see Section 2) rather than assumed.
GRD_PATHS = {
    2021: (fr"{DATA_ROOT}\GRD\GRD_PUBLICO_2021.txt", "latin1"),
    2022: (fr"{DATA_ROOT}\GRD\GRD_PUBLICO_EXTERNO_2022.txt", "utf-16"),
    2023: (fr"{DATA_ROOT}\GRD\GRD_PUBLICO_2023.txt", "utf-16"),
    2024: (fr"{DATA_ROOT}\GRD\GRD_PUBLICO_2024.txt", "latin1"),
}

NEEDED_COLS = ["PREVISION", "IR_29301_PESO", "IR_29301_SEVERIDAD",
               "IR_29301_MORTALIDAD", "TIPO_INGRESO"]


## 2. Data quality exploration (Five Cs, before cleaning)

We check three things across the raw files before trusting them for any
KPI: whether column names are actually readable under the assumed encoding
(**Consistent**), whether any lines are malformed (**Clean**), and whether
the 2022 "EXTERNO" file represents the same population as the other years
(**Conformed**).

In [3]:
# --- Consistent: confirm the assumed encoding actually yields the expected
# column names for every year. A file with the wrong encoding still "loads"
# with the right column COUNT but garbled names, which silently breaks any
# downstream usecols filter instead of raising an error.
print("=== Column consistency check ===")
for year, (path, enc) in GRD_PATHS.items():
    with open(path, "r", encoding=enc) as f:
        header = f.readline()
    cols = header.strip().split("|")
    missing = [c for c in NEEDED_COLS if c not in cols]
    print(f"{year} ({enc}): {len(cols)} columns | "
          f"missing needed columns: {missing if missing else 'none'}")


=== Column consistency check ===
2021 (latin1): 129 columns | missing needed columns: none
2022 (utf-16): 129 columns | missing needed columns: none
2023 (utf-16): 129 columns | missing needed columns: none
2024 (latin1): 129 columns | missing needed columns: none


In [4]:
# --- Clean: count malformed lines (wrong number of "|" separators) per
# year. A text field containing a literal "|" character breaks the column
# count for that single line; this is expected to be rare.
print("=== Malformed line check ===")
for year, (path, enc) in GRD_PATHS.items():
    with open(path, "r", encoding=enc) as f:
        header = f.readline()
        expected_pipes = header.count("|")
        n_bad = 0
        for line in f:
            if line.count("|") != expected_pipes:
                n_bad += 1
    print(f"{year}: {n_bad} malformed line(s) out of the file")


=== Malformed line check ===
2021: 0 malformed line(s) out of the file
2022: 1 malformed line(s) out of the file
2023: 0 malformed line(s) out of the file
2024: 0 malformed line(s) out of the file


In [5]:
# --- Conformed: confirm the 2022 "EXTERNO" file represents the same
# underlying population as the other years, not a different cohort (e.g. a
# pandemic-era private-provider referral agreement), before including it in
# a multi-year trend.
print("=== PREVISION distribution: 2022 (EXTERNO) vs 2024 (reference) ===")
for year in [2022, 2024]:
    path, enc = GRD_PATHS[year]
    is_utf16 = (enc == "utf-16")
    df_prev = pd.read_csv(
        path, sep="|", encoding=enc, usecols=["PREVISION"], dtype=str,
        engine=("python" if is_utf16 else "c"), on_bad_lines="skip",
    )
    print(f"\n--- {year} ---")
    print((df_prev["PREVISION"].value_counts(normalize=True) * 100).round(2))


=== PREVISION distribution: 2022 (EXTERNO) vs 2024 (reference) ===

--- 2022 ---
PREVISION
FONASA INSTITUCIONAL - (MAI) B    47.13
FONASA INSTITUCIONAL - (MAI) A    23.33
FONASA INSTITUCIONAL - (MAI) D    13.91
FONASA INSTITUCIONAL - (MAI) C    11.72
FONASA LIBRE ELECCIÓN (FMLE_B)     1.00
PARTICULAR                         0.81
ISAPRE                             0.80
FONASA LIBRE ELECCIÓN (FMLE_D)     0.62
FONASA LIBRE ELECCIÓN (FMLE_C)     0.30
DIPRECA                            0.24
CAPREDENA                          0.08
CAJA DE PREVISIÓN FFAA (SISA)      0.06
DESCONOCIDO                        0.01
Name: proportion, dtype: float64

--- 2024 ---
PREVISION
FONASA INSTITUCIONAL - (MAI) B    48.74
FONASA INSTITUCIONAL - (MAI) A    20.80
FONASA INSTITUCIONAL - (MAI) D    14.70
FONASA INSTITUCIONAL - (MAI) C    12.81
FONASA LIBRE ELECCIÓN (FMLE_B)     0.90
FONASA LIBRE ELECCIÓN (FMLE_D)     0.55
ISAPRE                             0.54
PARTICULAR                         0.33
FONASA LIBRE

**Finding:** the 2022 (EXTERNO) file shows the same FONASA-dominated
distribution as 2024, with only a modest increase in the PRIVATE share
(consistent with normal year-to-year variation, not a structurally
different population). This supports including 2022 in the multi-year
trend below. See the W1→C1 process record for the reasoning that ruled out
the "pandemic private-referral agreement" alternative explanation.

## 3. Cleaning pipeline

Insurance categories are grouped into FONASA / PRIVATE / MILITARY.
Ambulatory-surgery cases with no real clinical grouping assigned by the GRD
grouper (`IR_29301_SEVERIDAD == '0'` and `IR_29301_MORTALIDAD == '0'`,
confirmed to be ~100% Cirugía Mayor Ambulatoria cases) are excluded, since
severity/mortality scales are not meaningful for same-day ambulatory
procedures.

In [6]:
INSURANCE_MAP = {
    "FONASA INSTITUCIONAL - (MAI) A": "FONASA", "FONASA INSTITUCIONAL - (MAI) B": "FONASA",
    "FONASA INSTITUCIONAL - (MAI) C": "FONASA", "FONASA INSTITUCIONAL - (MAI) D": "FONASA",
    "FONASA LIBRE ELECCIÓN (FMLE_B)": "FONASA", "FONASA LIBRE ELECCIÓN (FMLE_C)": "FONASA",
    "FONASA LIBRE ELECCIÓN (FMLE_D)": "FONASA",
    "ISAPRE": "PRIVATE", "PARTICULAR": "PRIVATE",
    "DIPRECA": "MILITARY", "CAPREDENA": "MILITARY", "CAJA DE PREVISIÓN FFAA (SISA)": "MILITARY",
    "DESCONOCIDO": None,
}


In [7]:
# Confirm the actual TIPO_INGRESO categories before assuming any exact
# string for an "urgent admission" flag - do not hardcode a guessed label.
path, enc = GRD_PATHS[2024]
df_check = pd.read_csv(
    path, sep="|", encoding=enc, usecols=["TIPO_INGRESO"], dtype=str,
    engine="c", on_bad_lines="skip",
)
print(df_check["TIPO_INGRESO"].value_counts(dropna=False))


TIPO_INGRESO
URGENCIA       552102
PROGRAMADA     388739
OBSTETRICA     144929
DESCONOCIDO        43
Name: count, dtype: int64


**Action based on the result above:** define the "urgent admission"
flag using a case-insensitive match on whichever category name(s) actually
contain "URGEN" in this file, rather than a single hardcoded literal —
confirm the match list against the printed counts before trusting the KPI
below.

In [8]:
URGENT_LABELS = [v for v in df_check["TIPO_INGRESO"].dropna().unique()
                  if "URGEN" in v.upper()]
print("Categories treated as 'urgent':", URGENT_LABELS)


Categories treated as 'urgent': ['URGENCIA']


In [9]:
def load_and_clean_grd(year, path, encoding):
    """
    Loads one year of GRD data and applies the validated cleaning steps:
    - reads only the needed columns as strings (avoids a pandas
      mixed-dtype/usecols bug seen when letting the C engine infer types)
    - parses PESO (comma decimal -> float)
    - groups PREVISION into FONASA / PRIVATE / MILITARY
    - excludes ambulatory-surgery records with no real clinical grouping
    - flags admissions matching any label in URGENT_LABELS
    Returns the cleaned dataframe plus a stats dict for the process record.
    """
    is_utf16 = (encoding == "utf-16")

    df = pd.read_csv(
        path, sep="|", encoding=encoding, usecols=NEEDED_COLS, dtype=str,
        engine=("python" if is_utf16 else "c"), on_bad_lines="skip",
    )
    n_raw = len(df)

    # Clean: partial duplicate check, limited to the columns actually
    # loaded (no patient identifier is loaded here, so this flags exact
    # duplicate rows on these 5 fields only - a coarse signal, not a full
    # record-level duplicate check).
    n_duplicates = df.duplicated().sum()

    df["PESO_num"] = pd.to_numeric(
        df["IR_29301_PESO"].str.replace(",", ".", regex=False), errors="coerce"
    )
    df["insurance_group"] = df["PREVISION"].map(INSURANCE_MAP)

    no_grouping = (df["IR_29301_SEVERIDAD"] == "0") & (df["IR_29301_MORTALIDAD"] == "0")

    valid = df[df["insurance_group"].notna() & ~no_grouping &
               (df["IR_29301_SEVERIDAD"] != "DESCONOCIDO")].copy()
    valid["high_severity"] = valid["IR_29301_SEVERIDAD"] == "3"
    valid["urgent_admission"] = valid["TIPO_INGRESO"].isin(URGENT_LABELS)

    stats = {
        "year": year, "n_raw": n_raw, "n_duplicate_rows": n_duplicates,
        "n_unknown_insurance": df["insurance_group"].isna().sum(),
        "n_no_clinical_grouping": no_grouping.sum(),
        "n_valid": len(valid),
    }
    return valid, stats


## 4. Research question — how it evolved

**Original framing (W1):** compare complexity between public and private
hospitals. Rejected once we confirmed that Chile's open GRD data only
covers the public network (SNSS) — private hospitals are not required to
publish case-mix data (see the W1 report and process record for that
decision, backed by a FONASA institutional presentation, Aug. 2024, and a
*La Tercera* article on a pandemic-era private-referral GRD agreement,
2021).

**First C1 framing:** within the public network, do FONASA beneficiaries
show higher severity than PRIVATE (Isapre/Particular) beneficiaries?

**Refinement (this note):** patients with private insurance who appear in
this dataset are a self-selected subsample — they have private coverage
but still ended up in a public hospital anyway (likely due to an
emergency, lack of coverage for that specific procedure, or proximity). We
cannot observe privately-insured patients who never use the public network
at all.

**Refined question:** among privately-insured patients who *do* end up in
the public network, is their severity profile lower than FONASA
beneficiaries (suggesting they only use public care for minor/incidental
cases) or similar/higher (suggesting they only arrive there for
emergencies the private network could not resolve in time)?

**Key limitation to keep in the report:** this does **not** allow a full
"public vs. private system" complexity comparison — only a comparison of
who, among those who do reach the public network, arrives with higher or
lower severity depending on their insurance type.

## 5. KPI calculation across years (2021-2024)

In [10]:
yearly_data = {}
results, stats_log, admission_results = [], [], []

print("Loading all years...")
for year, (path, enc) in GRD_PATHS.items():
    valid, stats = load_and_clean_grd(year, path, enc)
    yearly_data[year] = valid
    stats_log.append(stats)
    print(f"  {year}: {stats['n_valid']:,} valid rows "
          f"(of {stats['n_raw']:,} raw, {stats['n_duplicate_rows']} duplicate rows found)")

    kpi_weight = valid.groupby("insurance_group")["PESO_num"].mean()
    kpi_severity = valid.groupby("insurance_group")["high_severity"].mean() * 100
    kpi_urgent = valid.groupby("insurance_group")["urgent_admission"].mean() * 100
    n = valid["insurance_group"].value_counts()

    results.append({
        "year": year,
        "avg_weight_FONASA": kpi_weight.get("FONASA"),
        "avg_weight_PRIVATE": kpi_weight.get("PRIVATE"),
        "pct_high_sev_FONASA": kpi_severity.get("FONASA"),
        "pct_high_sev_PRIVATE": kpi_severity.get("PRIVATE"),
        "severity_gap_pp": kpi_severity.get("FONASA", 0) - kpi_severity.get("PRIVATE", 0),
        "n_FONASA": n.get("FONASA", 0), "n_PRIVATE": n.get("PRIVATE", 0),
    })
    admission_results.append({
        "year": year,
        "pct_urgent_FONASA": kpi_urgent.get("FONASA"),
        "pct_urgent_PRIVATE": kpi_urgent.get("PRIVATE"),
    })

summary = pd.DataFrame(results).sort_values("year").reset_index(drop=True)
stats_df = pd.DataFrame(stats_log).sort_values("year").reset_index(drop=True)
admission_summary = pd.DataFrame(admission_results).sort_values("year").reset_index(drop=True)

print("\n=== KPI 1 & 2: severity/weight by insurance group ===")
print(summary)
print("\n=== Load & cleaning stats (process record evidence) ===")
print(stats_df)
print("\n=== KPI 3: urgent admission share by insurance group ===")
print(admission_summary)


Loading all years...
  2021: 703,387 valid rows (of 816,909 raw, 798322 duplicate rows found)
  2022: 777,989 valid rows (of 932,840 raw, 913154 duplicate rows found)
  2023: 845,984 valid rows (of 1,039,587 raw, 1019985 duplicate rows found)
  2024: 873,711 valid rows (of 1,085,813 raw, 1066117 duplicate rows found)

=== KPI 1 & 2: severity/weight by insurance group ===
   year  avg_weight_FONASA  avg_weight_PRIVATE  pct_high_sev_FONASA  \
0  2021           1.189595            1.261929            26.920129   
1  2022           1.059461            0.986660            24.682114   
2  2023           1.054729            1.000686            24.194209   
3  2024           1.067676            1.044792            24.954563   

   pct_high_sev_PRIVATE  severity_gap_pp  n_FONASA  n_PRIVATE  
0             28.698315        -1.778186    687623      13060  
1             19.333863         5.348251    760887      13841  
2             19.330575         4.863634    831947      10636  
3             

## 6. Interpreting the 2021 anomaly

2021 shows an inverse pattern in severity (PRIVATE higher than FONASA,
gap of about -1.8pp) compared to 2022-2024. This coincides with a markedly
lower share of ambulatory-surgery cases with no clinical grouping (roughly
103K vs. 150-212K in other years) — consistent with elective surgeries
being postponed during the peak pandemic year, leaving a higher proportion
of genuine hospitalizations with assigned severity. This is treated as a
plausible contextual explanation, not a data defect.

Note the **admission-urgency KPI (Section 5) does not show this reversal**:
PRIVATE has a higher urgent-admission share than FONASA in **all four
years**, including 2021. This is useful for the report — it shows the two
KPIs capture different, complementary aspects of the same question, and
that the 2021 anomaly is specific to the severity measure, not a general
data problem for that year.

*Open observation, not further investigated given time constraints:* 2021
also shows an unusually high count of unmapped PREVISION categories
(11,601 vs. ~50 in other years).

## 7. Visualizations

In [11]:
plt.figure(figsize=(6, 4))
plt.plot(summary["year"], summary["pct_high_sev_FONASA"], marker="o", label="FONASA")
plt.plot(summary["year"], summary["pct_high_sev_PRIVATE"], marker="o", label="PRIVATE")
plt.ylabel("% high-severity cases")
plt.xlabel("Year")
plt.title("Share of high-severity GRD cases by insurance group (2021-2024)")
plt.legend()
plt.tight_layout()
plt.savefig("severity_gap_trend.png", dpi=150)
plt.close()
print("saved severity_gap_trend.png")


saved severity_gap_trend.png


In [12]:
plt.figure(figsize=(6, 4))
plt.plot(admission_summary["year"], admission_summary["pct_urgent_FONASA"], marker="o", label="FONASA")
plt.plot(admission_summary["year"], admission_summary["pct_urgent_PRIVATE"], marker="o", label="PRIVATE")
plt.ylabel("% admissions via urgent care")
plt.xlabel("Year")
plt.title("Share of urgent (non-elective) admissions by insurance group (2021-2024)")
plt.legend()
plt.tight_layout()
plt.savefig("admission_urgency_trend.png", dpi=150)
plt.close()
print("saved admission_urgency_trend.png")


saved admission_urgency_trend.png


## 8. Export reduced input/output tables

`data/input/`: per-year extracts limited to the columns actually used
(raw values, before grouping/filtering) — lets a reviewer reproduce our
cleaning steps from a much smaller file than the original.

`data/output/`: the cleaned per-year tables and the aggregated KPI tables
that back the report's figures and claims.

In [13]:
DATA_IN = Path("data/input")
DATA_OUT = Path("data/output")
DATA_IN.mkdir(parents=True, exist_ok=True)
DATA_OUT.mkdir(parents=True, exist_ok=True)

for year, (path, enc) in GRD_PATHS.items():
    is_utf16 = (enc == "utf-16")
    df_raw = pd.read_csv(
        path, sep="|", encoding=enc, usecols=NEEDED_COLS, dtype=str,
        engine=("python" if is_utf16 else "c"), on_bad_lines="skip",
    )
    df_raw.to_csv(DATA_IN / f"grd_{year}_reduced.csv", index=False, encoding="utf-8")
    print(f"input {year}: {len(df_raw):,} rows -> data/input/grd_{year}_reduced.csv")

for year, valid in yearly_data.items():
    valid[["PREVISION", "insurance_group", "PESO_num", "IR_29301_SEVERIDAD",
           "IR_29301_MORTALIDAD", "TIPO_INGRESO", "high_severity",
           "urgent_admission"]].to_csv(
        DATA_OUT / f"grd_{year}_cleaned.csv", index=False, encoding="utf-8"
    )

summary.to_csv(DATA_OUT / "summary_kpis_by_year.csv", index=False)
stats_df.to_csv(DATA_OUT / "load_stats_by_year.csv", index=False)
admission_summary.to_csv(DATA_OUT / "admission_summary_by_year.csv", index=False)

print("\nDone. Contents of data/:")
for p in sorted(Path("data").rglob("*.csv")):
    print(f"  {p}  ({p.stat().st_size / 1e6:.1f} MB)")


input 2021: 816,909 rows -> data/input/grd_2021_reduced.csv
input 2022: 932,840 rows -> data/input/grd_2022_reduced.csv
input 2023: 1,039,587 rows -> data/input/grd_2023_reduced.csv
input 2024: 1,085,813 rows -> data/input/grd_2024_reduced.csv

Done. Contents of data/:
  data\input\grd_2021_reduced.csv  (43.9 MB)
  data\input\grd_2022_reduced.csv  (50.8 MB)
  data\input\grd_2023_reduced.csv  (56.8 MB)
  data\input\grd_2024_reduced.csv  (59.3 MB)
  data\output\admission_summary_by_year.csv  (0.0 MB)
  data\output\grd_2021_cleaned.csv  (49.4 MB)
  data\output\grd_2022_cleaned.csv  (54.8 MB)
  data\output\grd_2023_cleaned.csv  (59.6 MB)
  data\output\grd_2024_cleaned.csv  (61.6 MB)
  data\output\load_stats_by_year.csv  (0.0 MB)
  data\output\summary_kpis_by_year.csv  (0.0 MB)


## Known limitations / not yet done

- No join with `Establecimientos` was performed — the question is fully
  answered with fields native to GRD alone; see the process record for the
  justification of why a join was not pursued.
- The duplicate check in Section 3 only covers the 5 loaded columns (no
  patient identifier loaded), so it is a coarse signal, not a full
  record-level duplicate check.
- The unmapped-PREVISION anomaly in 2021 (Section 6) was noted but not
  investigated further.
